# 01 · Data Exploration
#
**Question:** What exactly is in this dataset?
#
This notebook uses functions from `src/data.py` and `src/probabilities.py`.
All charts are interactive Plotly figures.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import plotly.express as px

from src import data as D
from src import probabilities as P
from src import plotting as plt


## Raw dataset overview

In [2]:
info = D.inspect()
info["season_counts"]


{'18/19': 380,
 '19/20': 380,
 '20/21': 380,
 '21/22': 380,
 '22/23': 380,
 '23/24': 380,
 '24/25': 380,
 '25/26': 380}

In [3]:
raw = D.load_raw()
print("Total rows (all seasons):", len(raw))
raw["season"].value_counts().sort_index().rename("matches")


Total rows (all seasons): 3040


season
18/19    380
19/20    380
20/21    380
21/22    380
22/23    380
23/24    380
24/25    380
25/26    380
Name: matches, dtype: int64

## Column identification
#
The raw files use the standard football-data.co.uk layout. The Bet365 1X2
odds appear as `B365H`, `B365D`, `B365A`. Bet365 **closing** odds appear as
`B365CH`, `B365CD`, `B365CA` in all files except 2018-19, which has no
Bet365 closing-odds fields. The result is `FTR` (H/D/A).

In [4]:
print("Bet365 columns present:")
for kind, cols in D.available_odds_columns(raw).items():
    print(f"  {kind}: {cols}")


Bet365 columns present:
  regular: {'B365D', 'B365A', 'B365H'}
  closing: {'B365CD', 'B365CH', 'B365CA'}


## Primary dataset
#
The primary analysis uses only seasons with true Bet365 closing odds and no
COVID disruption: **2020-21 .. 2025-26**. 2018-19 (no closing odds) and
2019-20 (COVID) are excluded.

In [5]:
primary = P.add_probability_columns(D.load_processed())
print("Primary rows:", len(primary))
primary["season"].value_counts().sort_index()


Primary rows: 2280


season
20/21    380
21/22    380
22/23    380
23/24    380
24/25    380
25/26    380
Name: count, dtype: int64

## Outcome frequencies

In [6]:
freq = primary["FTR"].value_counts(normalize=True).sort_index()
freq.round(4).rename("share").to_frame()


,share
FTR,
A,0.2838
D,0.2658
H,0.4504


## Odds distributions (closing, by outcome)

In [7]:
fig = plt.histogram(primary["B365CH"].dropna(), title="Bet365 H closing odds", x="odds", nbins=60)
fig.show()


In [8]:
for o, col in {"H": "B365CH", "D": "B365CD", "A": "B365CA"}.items():
    print(f"B365 {o} closing odds: median {primary[col].median():.2f}, "
          f"max {primary[col].max():.2f}, min {primary[col].min():.2f}")


B365 H closing odds: median 2.20, max 13.00, min 1.07
B365 D closing odds: median 3.50, max 12.00, min 2.50
B365 A closing odds: median 3.50, max 26.00, min 1.18


## Implied + normalised probability distributions

In [9]:
for o in ("H", "D", "A"):
    fig = plt.histogram(primary[f"p_{o}_norm"], title=f"Normalised p_{o}", x="probability", nbins=40)
    fig.show()


## Overround distribution

In [10]:
fig = plt.histogram(primary["overround"], title="Overround (closing book) distribution",
                    x="overround", nbins=40)
fig.show()
print(primary["overround"].describe().round(4))


count    2280.0000
mean        0.0560
std         0.0070
min         0.0290
25%         0.0517
50%         0.0560
75%         0.0596
max         0.1190
Name: overround, dtype: float64


## Overround by season

In [11]:
fig = plt.overround_by_season(primary)
fig.show()


## Missing / invalid values

In [12]:
odds_cols = ["B365CH", "B365CD", "B365CA", "FTR"]
print(primary[odds_cols].isna().sum())
print("\nInvalid odds (<= 1.0) — treatment: excluded from probability calc:")
for col in ["B365CH", "B365CD", "B365CA"]:
    n = (primary[col] <= 1.0).sum()
    print(f"  {col}: {n}")


B365CH    0
B365CD    0
B365CA    0
FTR       0
dtype: int64

Invalid odds (<= 1.0) — treatment: excluded from probability calc:
  B365CH: 0
  B365CD: 0
  B365CA: 0


## Key data-quality findings
#
* 8 seasons × 380 matches = 3,040 raw rows.
* Primary dataset: 2,280 matches (2020-21 .. 2025-26).
* 2018-19 lacks Bet365 **closing** odds (only regular B365) → excluded from primary.
* 2019-20 is COVID-disrupted → excluded from primary.
* Closing overround is small and fairly stable (median ≈ 5.6%, range ~3–12%).